# The recommender, tested once

Notebook 13 developed the pro-pick recommender on 2025 up to Masters Toronto, and chose its
settings and its headline method there. Those figures are optimistic by construction. This
notebook measures it on line-ups it has never been scored on: **the 2025 test half — 516
maps, 1,032 line-ups, 3 July to 5 October.**

The *win results* of those maps were seen in notebook 12. **None of their line-ups has ever
been used for recommending** — a different task, answered here for the first time.

**This notebook was committed to git before it was run**, like notebook 12. Everything below
was fixed in that commit.

## Decided in advance

1. **The methods and settings**, exactly as fixed at the end of notebook 13:

   | | Method | Recency half-life |
   |---|---|---|
   | **Headline** | **Gradient boosting** (`HistGradientBoostingClassifier`, defaults, `random_state=0`) | **45 days** |
   | | Classifier + map × agent columns | 21 days |
   | | Co-occurrence | 21 days |
   | | Classifier, plain | 45 days |
   | **The bar** | **Popularity — the naive rule** | 7 days |

2. **History.** All of 2024, plus 2025 up to Masters Toronto. Then walk-forward: each week
   of the test half is recommended using only line-ups played before that week, so earlier
   test-half weeks become history as they pass — as they would in real use.
3. **Measures**, as in notebook 13:
   - *top-1* and *top-3* — hide one agent from each line-up, give the map and the other four,
     and see where the hidden agent ranks; five questions per line-up;
   - *from scratch* — how many of the pros' five agents it picks from the map alone.
4. **Margin of error.** 5,000 resamples of whole matches, random seed 0. 95% ranges for the
   headline's top-1 and top-3, and for its **top-3 lead over the naive rule**.
5. **What counts as what, stated now:**
   - lead over the naive rule on top-3, with the bottom of its 95% range above zero →
     **beats the naive rule**;
   - a lead whose range includes zero → **ahead, but not distinguishable from the naive rule**;
   - no lead → reported as that, plainly.
6. **One breakdown, and only one: Corrode against every other map**, for the headline method.
   Corrode is a brand-new map, first played in this half, so the recommender starts with no
   history for it. The question is how it copes, which is stated here rather than found by
   looking. No other slices.
7. **Expected: lower than development.** The recency settings and the headline method were
   chosen on development data, and two methods were added after seeing results there. A drop
   is the honest cost of that.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)

from dataset import build_map_table, split_season
from recommend import Classifier, CoOccurrence, Popularity, lineup_table, summarise, walk_forward

SEED, RESAMPLES = 0, 5000
HEADLINE, BAR = "gradient boosting", "popularity (naive rule)"
METHODS = {
    "gradient boosting": lambda: Classifier(45, estimator=HistGradientBoostingClassifier(random_state=0)),
    "classifier + map x agent": lambda: Classifier(21, interactions=True),
    "co-occurrence": lambda: CoOccurrence(21),
    "classifier": lambda: Classifier(45),
    "popularity (naive rule)": lambda: Popularity(7),
}

## The line-ups

In [ ]:
learn_2025, test_2025 = split_season(build_map_table())
history = pd.concat([lineup_table(build_map_table("vct_2024")), lineup_table(learn_2025)], ignore_index=True)
to_score = lineup_table(test_2025)

print(f"history : {len(history):,} line-ups, {history.played_at.min():%d %b %Y} to {history.played_at.max():%d %b %Y}")
print(f"scored  : {len(to_score):,} line-ups from {to_score.match_id.nunique()} matches, "
      f"{to_score.played_at.min():%d %b} to {to_score.played_at.max():%d %b %Y}, "
      f"over {to_score.played_at.dt.to_period('W').nunique()} weeks")
print(f"          x 5 hidden agents = {len(to_score) * 5:,} complete-a-line-up questions")

new_maps = sorted(set(to_score["map"]) - set(history["map"]))
if new_maps:
    on_new_map = to_score["map"].isin(new_maps)
    print(f"\nmaps with no history at all: {new_maps}, first played "
          f"{to_score.loc[on_new_map, 'played_at'].min():%d %b}, {on_new_map.sum()} line-ups")
else:
    print("\nevery map scored here has history")

## Run every method once

Gradient boosting is refitted every week and is slow to query, so this cell takes several
minutes.

In [ ]:
runs = {name: walk_forward(history, to_score, make) for name, make in METHODS.items()}

## The result

In [ ]:
results = pd.DataFrame({name: summarise(r) for name, r in runs.items()}).T
print(f"{len(to_score):,} test line-ups.  Picking at random from the ~23 agents left gets top-3 about 13%.\n")
print(results.to_string(formatters={"top-1": "{:.1%}".format, "top-3": "{:.1%}".format,
                                    "from scratch, of 5": "{:.2f}".format}))

## The margin of error

In [ ]:
def per_lineup(result, cutoff):
    return (np.vstack(result["ranks"].to_numpy()) <= cutoff).mean(axis=1)

match_of = runs[HEADLINE]["match_id"].to_numpy()
match_ids = np.unique(match_of)
rows_of = {m: np.flatnonzero(match_of == m) for m in match_ids}
rng = np.random.default_rng(SEED)
draws = [np.concatenate([rows_of[m] for m in rng.choice(match_ids, len(match_ids))]) for _ in range(RESAMPLES)]

def interval(values):
    return np.percentile([values[d].mean() for d in draws], [2.5, 97.5])

for label, cutoff in [("top-3", 3), ("top-1", 1)]:
    v = per_lineup(runs[HEADLINE], cutoff)
    lo, hi = interval(v)
    print(f"{HEADLINE} {label}: {v.mean():.1%}   95% range {lo:.1%} to {hi:.1%}")

lead = per_lineup(runs[HEADLINE], 3) - per_lineup(runs[BAR], 3)
lead_lo, lead_hi = interval(lead)
print(f"\ntop-3 lead over the naive rule: {lead.mean() * 100:+.1f} points   "
      f"95% range {lead_lo * 100:+.1f} to {lead_hi * 100:+.1f}")

if lead.mean() > 0 and lead_lo > 0:
    verdict = "BEATS THE NAIVE RULE"
elif lead.mean() > 0:
    verdict = "AHEAD, BUT NOT DISTINGUISHABLE FROM THE NAIVE RULE"
else:
    verdict = "DOES NOT BEAT THE NAIVE RULE"
print(f"\nby the rule decided in advance: {verdict}")

## The one pre-registered breakdown: a brand-new map

In [ ]:
headline_run = runs[HEADLINE]
on_new = headline_run["map"].isin(new_maps)
groups = [(f"{', '.join(new_maps)} (new)", on_new), ("every other map", ~on_new)] if new_maps else []
if not groups:
    print("no brand-new map in the weeks scored -- nothing to break down")
for label, mask in groups:
    part = headline_run[mask]
    print(f"{label:18} {len(part):4} line-ups   " +
          "   ".join(f"{k} {v:.1%}" if k != "from scratch, of 5" else f"{k} {v:.2f}"
                     for k, v in summarise(part).items()))

## What this means

*Written after the cells above were run, in a separate commit. Everything above this line
was committed before running.*